# Day 06: The Single Neuron

**Goal:** Build your first actual neural network component — a neuron — and watch it learn logic gates.

### What's a neuron?

A neuron is almost exactly what we've been building:

```
Linear regression:  y = w1*x1 + w2*x2 + b         ← Days 2-3
Neuron:             y = activation(w1*x1 + w2*x2 + b)   ← Today
                        ^^^^^^^^^^
                        just added this!
```

The activation function is a simple non-linear transformation. It's what gives neural networks their power to learn complex patterns.

Today we'll build a neuron from scratch, train it on logic gates (AND, OR), and discover WHY a single neuron can't learn XOR — which motivates multi-layer networks (Day 07).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

## 1. Activation Functions — See What They Do

Let's visualize the three most common activation functions:

In [ ]:
# The three most common activation functions
z = torch.linspace(-5, 5, 100)

sigmoid_out = 1 / (1 + torch.exp(-z))     # squishes to (0, 1)
relu_out = torch.maximum(torch.tensor(0.0), z)  # max(0, z)
tanh_out = torch.tanh(z)                   # squishes to (-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(z.numpy(), sigmoid_out.numpy(), 'b-', linewidth=2)
axes[0].axhline(y=0, color='k', linewidth=0.5)
axes[0].axhline(y=1, color='gray', linewidth=0.5, linestyle='--')
axes[0].set_title('Sigmoid: 1 / (1 + e^-z)\nOutput: (0, 1)')
axes[0].set_xlabel('z (input to activation)')
axes[0].set_ylabel('output')
axes[0].grid(True, alpha=0.3)

axes[1].plot(z.numpy(), relu_out.numpy(), 'r-', linewidth=2)
axes[1].axhline(y=0, color='k', linewidth=0.5)
axes[1].set_title('ReLU: max(0, z)\nOutput: [0, ∞)')
axes[1].set_xlabel('z')
axes[1].grid(True, alpha=0.3)

axes[2].plot(z.numpy(), tanh_out.numpy(), 'g-', linewidth=2)
axes[2].axhline(y=0, color='k', linewidth=0.5)
axes[2].axhline(y=1, color='gray', linewidth=0.5, linestyle='--')
axes[2].axhline(y=-1, color='gray', linewidth=0.5, linestyle='--')
axes[2].set_title('Tanh\nOutput: (-1, 1)')
axes[2].set_xlabel('z')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key properties:")
print("  Sigmoid: 'probability' output — great for yes/no predictions")
print("  ReLU:    simple, fast, no vanishing gradients — default for hidden layers")
print("  Tanh:    centered at 0 — useful when you need negative outputs")

## 2. Why Activation Functions Matter — The "Stacking" Problem

Here's the critical insight: **without activation functions, stacking layers is pointless.**

Let's prove it:

```
Layer 1: z1 = 2*x + 3         (just a line)
Layer 2: z2 = 4*z1 + 5 = 4*(2x + 3) + 5 = 8x + 17   (still just a line!)
```

No matter how many linear layers you stack, you always get a line. You can only learn `y = mx + b`.

But add activation `σ`:
```
Layer 1: z1 = σ(2*x + 3)      (curved!)
Layer 2: z2 = 4*z1 + 5         (combination of curves)
```

Now you can learn anything. **Non-linearity is what makes neural networks powerful.**

In [ ]:
# Visualize: linear stacking vs non-linear stacking

x = torch.linspace(-3, 3, 100)

# 3 linear layers stacked (no activation)
linear_output = x
for i in range(3):
    linear_output = 1.5 * linear_output + 0.5
# Still a line!

# 3 layers WITH activation (ReLU)
nonlinear_output = x
for i in range(3):
    nonlinear_output = torch.maximum(torch.tensor(0.0), 1.5 * nonlinear_output - 0.5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(x.numpy(), x.numpy(), 'k--', alpha=0.3, label='Input (y=x)')
ax1.plot(x.numpy(), linear_output.numpy(), 'b-', linewidth=2, label='3 linear layers stacked')
ax1.set_title('WITHOUT activation: still just a line\n(stacking is useless)')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(x.numpy(), x.numpy(), 'k--', alpha=0.3, label='Input (y=x)')
ax2.plot(x.numpy(), nonlinear_output.numpy(), 'r-', linewidth=2, label='3 layers with ReLU')
ax2.set_title('WITH activation: non-linear curves\n(stacking adds power!)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("The left plot is still a line — 3 linear layers did nothing extra.")
print("The right plot has bends — now the network can learn curves!")
print("\nThis is WHY activation functions exist.")

## 3. Logic Gates — Our First Real Classification Problem

Logic gates are perfect for learning because:
- Only 4 possible inputs (all combinations of 0/1)
- Clear right/wrong answers
- Easy to visualize in 2D

### The three gates we'll work with:

```
AND GATE            OR GATE             XOR GATE
x1  x2  output      x1  x2  output      x1  x2  output
 0   0    0          0   0    0          0   0    0
 0   1    0          0   1    1          0   1    1
 1   0    0          1   0    1          1   0    1
 1   1    1          1   1    1          1   1    0
```

Let's set up the data:

In [ ]:
# Inputs are the same for all 3 gates
X = torch.tensor([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])

# Different labels for each gate
y_and = torch.tensor([0.0, 0.0, 0.0, 1.0])
y_or  = torch.tensor([0.0, 1.0, 1.0, 1.0])
y_xor = torch.tensor([0.0, 1.0, 1.0, 0.0])

# Visualize — this is KEY for understanding linear separability
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, y, title in zip(axes, [y_and, y_or, y_xor], ['AND', 'OR', 'XOR']):
    for i in range(4):
        color = 'red' if y[i] == 1 else 'blue'
        marker = 'o' if y[i] == 1 else 's'
        label = f"output = {int(y[i].item())}"
        ax.scatter(X[i, 0].item(), X[i, 1].item(), s=500, c=color, marker=marker, 
                   edgecolor='black', linewidth=2)
        ax.annotate(f'({int(X[i,0].item())},{int(X[i,1].item())})',
                    (X[i,0].item(), X[i,1].item()),
                    textcoords="offset points", xytext=(15, 15), fontsize=11)
    ax.set_title(f'{title} gate', fontsize=14, fontweight='bold')
    ax.set_xlabel('x1')
    ax.set_ylabel('x2')
    ax.set_xlim(-0.3, 1.3)
    ax.set_ylim(-0.3, 1.3)
    ax.grid(True, alpha=0.3)

# Add a dummy legend explaining the shapes
for ax in axes:
    ax.scatter([], [], s=200, c='red', marker='o', edgecolor='black', label='Output = 1')
    ax.scatter([], [], s=200, c='blue', marker='s', edgecolor='black', label='Output = 0')
    ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

print("LOOK CAREFULLY at each plot:")
print("  AND: the one red dot (1,1) is isolated — ANY line can separate it ✓")
print("  OR:  the one blue dot (0,0) is isolated — ANY line can separate it ✓")
print("  XOR: red dots at (0,1) and (1,0) are DIAGONALLY OPPOSITE — no line can separate!")

## 4. Build a Single Neuron From Scratch

Our neuron:
```
z = w1 * x1 + w2 * x2 + b        (weighted sum)
ŷ = sigmoid(z)                    (activation → probability between 0 and 1)
```

Loss: **Binary Cross-Entropy (BCE)** — the standard loss for 0/1 classification:
```
BCE = -[y * log(ŷ) + (1 - y) * log(1 - ŷ)]
```

Why BCE instead of MSE? BCE punishes confident-but-wrong predictions heavily, which is ideal for classification.

In [ ]:
# Reusable function to train a single neuron

def train_neuron(X, y, epochs=3000, lr=0.5, verbose=True):
    """
    Train a single neuron to learn the relationship between X and y.
    Returns final weights and loss history.
    """
    # Random starting weights
    w = torch.randn(2, requires_grad=True)  # w1 and w2
    b = torch.randn(1, requires_grad=True)  # bias
    
    optimizer = torch.optim.SGD([w, b], lr=lr)
    losses = []
    
    for epoch in range(epochs):
        # ---- FORWARD PASS ----
        # z = w1*x1 + w2*x2 + b  (for all 4 samples)
        z = X @ w + b               # shape: (4,)
        
        # Activation: sigmoid — converts z to probability
        y_pred = torch.sigmoid(z)
        
        # ---- LOSS: Binary Cross-Entropy ----
        # PyTorch has a stable version: binary_cross_entropy
        loss = torch.nn.functional.binary_cross_entropy(y_pred, y)
        losses.append(loss.item())
        
        # ---- BACKWARD + UPDATE ----
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if verbose and epoch % 500 == 0:
            print(f"Epoch {epoch:4d}: loss={loss.item():.4f}")
    
    return w.detach(), b.detach(), losses

## 5. Train on AND Gate

In [ ]:
print("Training a single neuron to learn AND:\n")
w_and, b_and, losses_and = train_neuron(X, y_and, epochs=3000, lr=1.0)

# Check predictions
with torch.no_grad():
    predictions = torch.sigmoid(X @ w_and + b_and)

print(f"\nFinal predictions (AND):")
print(f"{'Input':>10} | {'Target':>6} | {'Predicted':>9} | {'Correct?':>8}")
print("-" * 50)
for i in range(4):
    x1, x2 = X[i, 0].item(), X[i, 1].item()
    target = y_and[i].item()
    pred = predictions[i].item()
    correct = "✓" if round(pred) == target else "✗"
    print(f"  ({x1:.0f}, {x2:.0f})   | {target:>6.0f} | {pred:>9.3f} | {correct:>8}")

print(f"\nLearned weights: w1={w_and[0].item():.2f}, w2={w_and[1].item():.2f}, b={b_and.item():.2f}")

### Visualizing the decision boundary

The neuron learns a line `w1*x1 + w2*x2 + b = 0` that separates the two classes. Let's plot it:

In [ ]:
# Helper function to plot decision boundary for a trained neuron

def plot_neuron_boundary(X, y, w, b, title, loss_history):
    """Plot the decision boundary learned by the neuron."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Left: data + decision boundary
    # Create a grid and compute neuron output at every point
    x_grid = torch.linspace(-0.3, 1.3, 200)
    y_grid = torch.linspace(-0.3, 1.3, 200)
    xx, yy = torch.meshgrid(x_grid, y_grid, indexing='xy')
    grid_points = torch.stack([xx.flatten(), yy.flatten()], dim=1)
    
    with torch.no_grad():
        z_grid = torch.sigmoid(grid_points @ w + b).reshape(xx.shape)
    
    # Contour plot showing the probability
    contour = ax1.contourf(xx.numpy(), yy.numpy(), z_grid.numpy(), 
                            levels=20, cmap='RdBu_r', alpha=0.6)
    plt.colorbar(contour, ax=ax1, label='Predicted probability')
    
    # The decision boundary (where output = 0.5)
    ax1.contour(xx.numpy(), yy.numpy(), z_grid.numpy(), 
                levels=[0.5], colors='black', linewidths=2)
    
    # Data points
    for i in range(4):
        color = 'red' if y[i] == 1 else 'blue'
        marker = 'o' if y[i] == 1 else 's'
        ax1.scatter(X[i, 0].item(), X[i, 1].item(), s=400, c=color, marker=marker,
                    edgecolor='white', linewidth=3, zorder=5)
    
    ax1.set_title(f'{title}: learned decision boundary')
    ax1.set_xlabel('x1')
    ax1.set_ylabel('x2')
    ax1.grid(True, alpha=0.3)
    
    # Right: loss curve
    ax2.plot(loss_history, 'b-', linewidth=1.5)
    ax2.set_title(f'{title}: loss over training')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Loss (BCE)')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

plot_neuron_boundary(X, y_and, w_and, b_and, 'AND', losses_and)

print("The BLACK LINE is the decision boundary (where probability = 0.5)")
print("BLUE region: neuron outputs 0 (low probability)")
print("RED region: neuron outputs 1 (high probability)")
print("\nThe line separates the single (1,1) from the three 0-outputs. ✓")

## 6. Train on OR Gate

In [ ]:
print("Training a single neuron to learn OR:\n")
w_or, b_or, losses_or = train_neuron(X, y_or, epochs=3000, lr=1.0)

with torch.no_grad():
    predictions = torch.sigmoid(X @ w_or + b_or)

print(f"\nFinal predictions (OR):")
print(f"{'Input':>10} | {'Target':>6} | {'Predicted':>9} | {'Correct?':>8}")
print("-" * 50)
for i in range(4):
    x1, x2 = X[i, 0].item(), X[i, 1].item()
    target = y_or[i].item()
    pred = predictions[i].item()
    correct = "✓" if round(pred) == target else "✗"
    print(f"  ({x1:.0f}, {x2:.0f})   | {target:>6.0f} | {pred:>9.3f} | {correct:>8}")

plot_neuron_boundary(X, y_or, w_or, b_or, 'OR', losses_or)

print("Same story as AND — one line cleanly separates the classes.")

## 7. Try XOR — And Watch It Fail!

This is the important part. Let's try to learn XOR with a single neuron:

In [ ]:
print("Training a single neuron to learn XOR (spoiler: it won't work)\n")
w_xor, b_xor, losses_xor = train_neuron(X, y_xor, epochs=5000, lr=1.0)

with torch.no_grad():
    predictions = torch.sigmoid(X @ w_xor + b_xor)

print(f"\nFinal predictions (XOR):")
print(f"{'Input':>10} | {'Target':>6} | {'Predicted':>9} | {'Correct?':>8}")
print("-" * 50)
for i in range(4):
    x1, x2 = X[i, 0].item(), X[i, 1].item()
    target = y_xor[i].item()
    pred = predictions[i].item()
    correct = "✓" if round(pred) == target else "✗"
    print(f"  ({x1:.0f}, {x2:.0f})   | {target:>6.0f} | {pred:>9.3f} | {correct:>8}")

plot_neuron_boundary(X, y_xor, w_xor, b_xor, 'XOR (FAILS!)', losses_xor)

print("Look at the predictions — they're all around 0.5 (totally uncertain)!")
print("The loss is stuck around 0.69 — that's what you get when you guess 50/50.")
print("\nNo matter where you draw a STRAIGHT LINE, you can't separate:")
print("  Red dots at (0,1) and (1,0) — these should output 1")
print("  Blue dots at (0,0) and (1,1) — these should output 0")
print("\nThey're on DIAGONAL corners. A single neuron is STUCK.")

### Why did XOR fail?

A single neuron computes: `y = sigmoid(w1*x1 + w2*x2 + b)`

The decision boundary is the line where `w1*x1 + w2*x2 + b = 0`. Solving for x2:
```
x2 = -(w1/w2)*x1 - b/w2
```

That's a straight line. **A single neuron can only draw ONE straight line.**

For XOR, no straight line works:

```
  x2
   |
 1 | 1 ← red           0 ← blue
   |         ↘       ↗
   |          ? — no line between these
   |         ↗       ↘
 0 | 0 ← blue          1 ← red
   +-------------------→ x1
     0                 1
```

The fix: stack MORE neurons. Two neurons can draw two lines, and combining them creates non-linear regions. That's what we'll do in Day 07.

This exact problem delayed AI research for ~20 years (the "AI winter" of the 1960s-70s) until researchers figured out how to train multi-layer networks.

---

## 8. Comparing All Three Gates Side by Side

In [ ]:
# Compare all 3 loss curves — one plot says it all

plt.figure(figsize=(10, 5))
plt.plot(losses_and, 'g-', label='AND  (converges ✓)', linewidth=2)
plt.plot(losses_or,  'b-', label='OR   (converges ✓)', linewidth=2)
plt.plot(losses_xor, 'r-', label='XOR  (stuck at ~0.69 ✗)', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss (BCE)')
plt.title('Single neuron learning 3 logic gates')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.show()

print("AND and OR drop to near 0 — the neuron LEARNED.")
print("XOR plateaus at 0.69 — that's the loss of random guessing (log(2) ≈ 0.693)")
print("\nThis tells us a single neuron has fundamental limits.")
print("Tomorrow (Day 07): we stack neurons and crack XOR.")

## 9. The Same Thing Using `nn.Module` (Preview of Day 9)

Writing `w = torch.randn(2, requires_grad=True)` manually is tedious. PyTorch provides `nn.Linear` which is literally a neuron (well, without the activation).

Here's the same AND gate training but with PyTorch's building blocks — this is the STANDARD way to write models from now on:

In [ ]:
import torch.nn as nn

# A single neuron using PyTorch's built-in building blocks
class SingleNeuron(nn.Module):
    def __init__(self, n_inputs):
        super().__init__()
        # nn.Linear(in_features, out_features) IS a neuron (minus the activation)
        # It automatically creates weights and bias with requires_grad=True
        self.linear = nn.Linear(n_inputs, 1)
    
    def forward(self, x):
        z = self.linear(x)          # weighted sum + bias
        return torch.sigmoid(z)      # activation
        # Note: we'll usually return z (logits) and use BCEWithLogitsLoss for stability

# Create the model
model = SingleNeuron(n_inputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=1.0)

# Training loop — same pattern, cleaner code
losses = []
for epoch in range(3000):
    # Forward: model(x) calls our forward() method
    y_pred = model(X).squeeze()  # squeeze removes extra dim: (4,1) → (4,)
    loss = nn.functional.binary_cross_entropy(y_pred, y_and)
    losses.append(loss.item())
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Check results
with torch.no_grad():
    predictions = model(X).squeeze()

print("AND gate with nn.Module:")
for i in range(4):
    x1, x2 = X[i, 0].item(), X[i, 1].item()
    target = y_and[i].item()
    pred = predictions[i].item()
    print(f"  ({x1:.0f}, {x2:.0f}) → predicted={pred:.3f}, target={target:.0f}")

print(f"\nFinal loss: {losses[-1]:.4f}")
print(f"\nCompare this code to our scratch version — MUCH cleaner.")
print("From Day 09 onwards, we'll always use nn.Module.")

---

## Exercises

1. **NAND gate** (opposite of AND): `[1, 1, 1, 0]`. Can a single neuron learn it? Try it.

2. **Different activations:** Modify `train_neuron` to use `tanh` instead of `sigmoid`. Do you need to change the loss function? (Hint: tanh outputs -1 to 1, but BCE expects 0 to 1)

3. **Learning rate sweep:** Train AND with lr=0.01, 0.1, 1.0, 10.0. Plot loss curves for each. What do you notice?

4. **Visualize activation:** Before and after training, plot `sigmoid(X @ w + b)` for 100 random points in [0,1]². How does the decision boundary evolve?

5. **Break the XOR record:** Try harder to make a single neuron learn XOR. More epochs? Different optimizer? Bigger learning rate? You'll find it's IMPOSSIBLE — and this proves the need for multi-layer networks.

---

## Key Takeaways

### The neuron equation:

```
z = w1*x1 + w2*x2 + b
y = activation(z)
```

### Why activation matters:

Without activation → stacking is just one line (useless).
With activation → stacking creates arbitrary complex curves (powerful).

### Linear separability:

A single neuron can only solve problems where a single straight line separates classes.
- AND ✓  (one dot isolated)
- OR ✓  (one dot isolated)
- XOR ✗ (diagonal pattern — no line works!)

### The limitation:

A single neuron = one linear decision boundary. For XOR (and most real problems), you need **multiple neurons** creating non-linear boundaries. That's why deep networks exist.

### Looking ahead:

```
Day 06: ONE neuron     → solves AND, OR (simple problems)
Day 07: TWO layers    → solves XOR (stacked = non-linear)
Day 08-20: Training loops, bigger networks
Day 21+:  Transformers (thousands of neurons in each layer)
```

**Tomorrow:** We stack neurons into layers and finally crack XOR!